In [ ]:
import pandas as pd
import io

In [ ]:
def parse_log_file(log_file, product):
    with open(log_file, "r") as file:
        file_content = file.read()
    sections = file_content.split("Sandbox logs:")[1].split("Activities log:")
    activities_log = sections[1].split("Trade History:")[0]
    df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_json = pd.json_normalize(
        json.loads(sections[1].split("Trade History:")[1])
    ).to_json()
    sandbox_logs = []
    logs_data = sections[0].strip()
    start_index = 0
    while start_index < len(logs_data):
        if logs_data[start_index] == "{":
            end_index = logs_data.find("}", start_index) + 1
            log_entry = logs_data[start_index:end_index]
            sandbox_logs.append(json.loads(log_entry))
            start_index = end_index
        else:
            start_index += 1
    df_sandbox = pd.DataFrame(sandbox_logs)
    if "lambdaLog" in df_sandbox.columns:
        if df_sandbox["lambdaLog"].str.contains("IMPLIED_BID").any():
            implied_bid = pd.to_numeric(df_sandbox["lambdaLog"].str.extract(r"IMPLIED_BID: (\d+\.\d+)")[0], errors='coerce')
            implied_bid_df = pd.DataFrame({"timestamp": df_sandbox["timestamp"], "implied_bid": implied_bid, "product": "ORCHIDS"})
            df = pd.merge(df, implied_bid_df, on=["timestamp", "product"], how="left")
        if df_sandbox["lambdaLog"].str.contains("IMPLIED_ASK").any():
            implied_ask = pd.to_numeric(df_sandbox["lambdaLog"].str.extract(r"IMPLIED_ASK: (\d+\.\d+)")[0], errors='coerce')
            implied_ask_df = pd.DataFrame({"timestamp": df_sandbox["timestamp"], "implied_ask": implied_ask, "product": "ORCHIDS"})
            df = pd.merge(df, implied_ask_df, on=["timestamp", "product"], how="left")
        if df_sandbox["lambdaLog"].str.contains("FOREIGN_ASK").any():
            foreign_ask = pd.to_numeric(df_sandbox["lambdaLog"].str.extract(r"FOREIGN_ASK: (\d+\.\d+)")[0], errors='coerce')
            foreign_ask_df = pd.DataFrame({"timestamp": df_sandbox["timestamp"], "foreign_ask": foreign_ask, "product": "ORCHIDS"})
            df = pd.merge(df, foreign_ask_df, on=["timestamp", "product"], how="left")
        if df_sandbox["lambdaLog"].str.contains("FOREIGN_BID").any():
            foreign_bid = pd.to_numeric(df_sandbox["lambdaLog"].str.extract(r"FOREIGN_BID: (\d+\.\d+)")[0], errors='coerce')
            foreign_bid_df = pd.DataFrame({"timestamp": df_sandbox["timestamp"], "foreign_bid": foreign_bid, "product": "ORCHIDS"})
            df = pd.merge(df, foreign_bid_df, on=["timestamp", "product"], how="left")
    return df, trade_json, df_sandbox.to_json()

In [ ]:
df, trades, _= parse_log_file('automated_test_value_3/run_0/submission.log', 'ORCHIDS')

In [ ]:
df[df['product'] == 'ORCHIDS']

In [ ]:
df = df[df['product'] == 'ORCHIDS'].reset_index(drop=True)

In [ ]:
df

In [ ]:
for i in range(0, 13):
    level = i - 6
    _, trades_level_json, _ = parse_log_file(f'automated_test_value_3/run_{i}/submission.log', 'ORCHIDS')
    trades_level_df = pd.read_json(trades_level_json)
    trades_level_df = trades_level_df[trades_level_df['symbol'] == 'ORCHIDS']
    trades_level_df = trades_level_df[['timestamp', 'price', 'quantity']]
    trades_level_df = trades_level_df.rename(columns={'price': f'price_level_{level}', 'quantity': f'quantity_level_{level}'})
    
    # Group by timestamp and aggregate price and quantity
    trades_level_df = trades_level_df.groupby('timestamp').agg({
        f'price_level_{level}': 'min',
        f'quantity_level_{level}': 'sum'
    }).reset_index()
    
#     display(trades_level_df)
    df = df.merge(trades_level_df, on="timestamp", how='left')
#     display(df)

In [ ]:
df

In [ ]:
df[([f"price_level_{i}" for i in range(-6, 7)] + [f"quantity_level_{i}" for i in range(-6, 7)])].mean()

In [ ]:
# Create a boolean mask for rows where any quantity level is greater than or equal to 90
mask = df[[f'quantity_level_{i}' for i in range(7)]].ge(100).any(axis=1)

# Create a new dataframe with the filtered rows
df_limit = df[mask]

In [ ]:
df_limit

In [ ]:
df_limit[[f'quantity_level_{i}' for i in range(0, 7)]]

In [ ]:
# Create a new dataframe with the quantity level columns
quantity_cols = [f'quantity_level_{i}' for i in range(7)]
quantity_df = df_limit[quantity_cols]

# Replace values not equal to 100 with NaN
quantity_df = quantity_df.where(quantity_df == 100)

# Find the index of the first column with value 100 for each row
best_level = quantity_df.apply(lambda row: row.first_valid_index(), axis=1)

# Extract the level number from the column name
df_limit['best_level'] = best_level.str.extract('(\d+)', expand=False).astype(float)

In [ ]:

df_limit['best_level'].value_counts()

In [ ]:
df_limit.columns

In [ ]:
df_limit['best_level'] = -df_limit['best_level']

In [ ]:
import plotly.graph_objects as go

# Create a new figure
fig = go.Figure()

# Define the columns to plot
columns_to_plot = ['bid_price_1', 'bid_price_2', 'bid_price_3',
                   'ask_price_1', 'ask_price_2', 'ask_price_3',
                   'mid_price', 'implied_bid', 'implied_ask',
                   'foreign_ask', 'foreign_bid']

# Add traces for each column
for col in columns_to_plot:
    fig.add_trace(go.Scatter(x=df_limit['timestamp'], y=df_limit[col], name=col))

# Add a trace for best_level on a separate y-axis
fig.add_trace(go.Scatter(x=df_limit['timestamp'], y=df_limit['best_level'], name='Best Level', yaxis='y2'))

# Configure the layout
fig.update_layout(
    title='Price and Best Level vs. Timestamp',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='Best Level', overlaying='y', side='right', range=[-2, 0], tickvals=[-2, -1, 0,]),
    legend=dict(x=1.1, y=1, orientation='v'),
    width=800,
    height=500
)

# Display the plot
fig.show()

In [ ]:
df_limit['best_level'] = -df_limit['best_level']

import plotly.graph_objects as go

# Create a new figure
fig = go.Figure()

# Define the columns to plot
columns_to_plot = ['bid_price_1', 'bid_price_2', 'bid_price_3',
                   'ask_price_1', 'ask_price_2', 'ask_price_3',
                   'mid_price', 'implied_bid', 'implied_ask',
                   'foreign_ask', 'foreign_bid']

# Add traces for each column
for col in columns_to_plot:
    if 'bid_price' in col:
        bid_volume_col = col.replace('price', 'volume')
        hovertemplate = f'Timestamp: %{{x}}<br>{col}: %{{y}}<br>{bid_volume_col}: %{{customdata}}'
        customdata = df_limit[bid_volume_col]
    elif 'ask_price' in col:
        ask_volume_col = col.replace('price', 'volume')
        hovertemplate = f'Timestamp: %{{x}}<br>{col}: %{{y}}<br>{ask_volume_col}: %{{customdata}}'
        customdata = df_limit[ask_volume_col]
    else:
        hovertemplate = f'Timestamp: %{{x}}<br>{col}: %{{y}}'
        customdata = None

    fig.add_trace(go.Scatter(x=df_limit['timestamp'], y=df_limit[col], name=col,
                             hovertemplate=hovertemplate, customdata=customdata))

# Add a trace for best_level on a separate y-axis
fig.add_trace(go.Scatter(x=df_limit['timestamp'], y=df_limit['best_level'], name='Best Level', yaxis='y2',
                         hovertemplate='Timestamp: %{x}<br>Best Level: %{y}'))

# Configure the layout
fig.update_layout(
    title='Price and Best Level vs. Timestamp',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='Best Level', overlaying='y', side='right', range=[-2, 0], tickvals=[-2, -1, 0]),
    legend=dict(x=1.1, y=1, orientation='v'),
    width=800,
    height=500,
    hovermode='x unified',
    hoverlabel=dict(font=dict(size=8))  # Adjust the font size of the hover labels
)

# Display the plot
fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define the columns for bid and ask volumes
bid_volume_cols = ['bid_volume_1', 'bid_volume_2', 'bid_volume_3']
ask_volume_cols = ['ask_volume_1', 'ask_volume_2', 'ask_volume_3']

# Create a figure and axes for the subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))

# Flatten the axes array for easier indexing
axes = axes.flatten()

# Plot the histograms for bid volumes
for i, col in enumerate(bid_volume_cols):
    sns.histplot(data=df_limit, x=col, bins=20, ax=axes[i])
    axes[i].set_title(f'Histogram of {col}')
    axes[i].set_xlabel('Volume')
    axes[i].set_ylabel('Count')

# Plot the histograms for ask volumes
for i, col in enumerate(ask_volume_cols):
    sns.histplot(data=df_limit, x=col, bins=20, ax=axes[i+3])
    axes[i+3].set_title(f'Histogram of {col}')
    axes[i+3].set_xlabel('Volume')
    axes[i+3].set_ylabel('Count')

# Adjust the spacing between subplots
plt.tight_layout()

# Display the plot
plt.show()